# Construction Materials Cost Planner - Analysis Notebook

**Goal:** estimate how much cement, sand and aggregate each concrete element needs (Module 1), then check what was actually bought and what it cost (Module 2).

The notebook uses the same code as `main.py`, so every number here can be reproduced from the menu.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import module1_quantities as m1
import module2_costs as m2

DATA = ROOT / "data"
registry = m1.ElementRegistry()
loaded, errors = registry.load_csv(DATA / "elements.csv")
print(loaded, "elements loaded | errors:", errors)

## 1. Required quantities (Module 1)

For each element: net volume = L x W x H, volume with waste = net x (1 + waste/100), then
300 kg cement, 0.5 m3 sand and 0.8 m3 aggregate per m3 (training rates).

In [ ]:
quantities = pd.DataFrame(registry.quantity_rows())
quantities

In [ ]:
pd.DataFrame(registry.quantities_by_site())

## 2. Cleaning the purchase data (pandas)

Rules applied: remove exact duplicate rows, remove rows with missing values, require quantity > 0,
reject unknown element/supplier IDs, and make sure every supplier/material pair has a price.
Rejected rows are **kept in a log** (`reports/rejected_records.csv`) instead of being silently deleted.

In [ ]:
analysis = m2.run_analysis(registry, DATA / "purchases.csv", DATA / "prices.csv")
print("rows read:", analysis["raw_rows"], "| kept:", len(analysis["clean"]),
      "| rejected:", len(analysis["rejected"]))
analysis["rejected"]

**Decision:** the 2 duplicated transactions (T009, T025) are removed because they are the same record twice.
The 3 rows with a missing quantity (T012, T014, T027) are removed rather than guessed - a made-up quantity would
create a made-up cost. Because of this, some elements show a shortage in the comparison below.

## 3. Purchase costs (merge + group)

`cost = quantity x unit_price`. Purchases are merged with prices on `supplier_id` + `material`,
and with elements on `element_id` (which gives the site).

In [ ]:
analysis["merged"].head(8)

In [ ]:
display_tables = {k: analysis[k] for k in ("cost_by_site", "cost_by_supplier", "cost_by_month")}
for name, table in display_tables.items():
    print(name); print(table.to_string(index=False)); print()

## 4. Required vs purchased

`difference = purchased - required`. Negative = **shortage**, positive = **surplus**.
Units are consistent: cement in kg, sand and aggregate in m3 (checked against the price file).

In [ ]:
comparison = analysis["comparison"]
print(comparison["status"].value_counts().to_string())
comparison

In [ ]:
# Shortages only, largest first (pandas filtering + sorting)
comparison[comparison["status"] == "Shortage"].sort_values("difference")

## 5. Price scenario (NumPy)

Baseline = the **lowest** available price of each material. The required quantities of all elements are
multiplied by the baseline prices, then one material's price is increased by a chosen percentage.

In [ ]:
baseline = m2.lowest_prices(analysis["prices"])
print(baseline.to_string())
table, summary = m2.price_increase_scenario(registry.total_required(), baseline, "Cement", 10)
print(table.to_string(index=False))
print(summary)

## 6. Charts

The four charts are saved as PNG files in `charts/`.

In [ ]:
paths = m2.create_charts(analysis, ROOT / "charts")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, path in zip(axes.ravel(), paths):
    ax.imshow(plt.imread(path)); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Conclusions

*Write your own conclusions here after running the notebook* (for example: which site costs most, which supplier
is used most, which materials are short, and how much the +10% cement price change adds to the total).